In [1]:
# '취업'에 대해 학습 시키켜록 한다
# 네이버 지식인에서 취업관 관련된 질문들을 가져와서 학습시키기 전 단계까지 처리하려고 한다

# 네이버 지식인에서 취업관 관련된 질문 200개를 추출해서 가져오는 작업
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 매개변수
# 변수명 : 타입 = 기본값
# def 함수() -> 리턴타입:
def get_kin_summary(keyword : str, page : int = 1)->pd.DataFrame:

	# URL 지정 
	url = 'https://kin.naver.com/search/list.naver'

	# url 뒤에 ?변수=값 형태로 만들기 위한 딕셔너리
	# 이 딕셔너리 이름은 반드시 parmas? X
	params = {
		'query' : keyword,
		'page' : page
	}
	# 왼쪽 params는 키워드 인자이기 때문 반드시 params
	response = requests.get(url, params=params)

	try:
		# 상태 코드가 200이면 아무것도 없음. 200이 아니면 예외 발생
		response.raise_for_status()
		# 확인용
		# print(response.text)
		
		# html 코드에서 원하는 요소를 탐색하기 위해 bs를 이용
		soup = BeautifulSoup(response.text, 'lxml') # lxml : html코드 , lxml-xml : 
		
		#._searchListTitleAnchor들을 선택
		# select, select_one : 선택자 기반으로 요소 선택(보통 html에서)
		# find_all, find : 태그 기반으로 요소 선택(보통 xml에서)
		link_elements = soup.select('._searchListTitleAnchor') # 최대 10개

		# 결과를 담을 리스트
		link_list, title_list = [], []

		# 반복문으로 선택한 요소를 활용
		for link_element in link_elements:
			# 확인용
			# print(link_element)
			title = link_element.text #요소 안에 있는태그들은 무시한 순수 글자들
			# 요소.get('속성') : 속성 값을 가져옴
			link = link_element.get('href')
			
			# 확인용
			# print(title)
			# print(link)
			
			# 추출한 데이터를 각 리스트에 추가
			link_list.append(link)
			title_list.append(title)

		# 확인용
		# print(link_list)
		# df : 변수이름(dataFrame)
		# 딕셔너리 : { '변수명' : 값, '변수명' : 값}
		# 데이터 처리를 위한 데이터 프레임 생성
		df = pd.DataFrame({
			'title' : title_list,
			'link' : link_list
		}) 
		# 확인용
		# print(df)
		return df
	except Exception as e:
		print(f'예외 발생 : {e}')
		return pd.DataFrame({
			'title' : [], 
			'link' : []
		})

In [2]:
import time

max_page = 20
keyword = '취업'

# 모든 결과를 담을 데이터프레임
total_df = pd.DataFrame({
	'title' : [], 
	'link' : []
})

# 200개 가져올 때까지 반복 => 기본 10 * 20페이지
for page in range(1, max_page + 1):

	# 키워드, 페이지를 이용하여 검색된 결과에서 제목과 내용을 추출
	df = get_kin_summary(keyword, page)

	# 확인용 
	# print(df)
	
	# 각 페이지 결과를 전체에 이어붙임
	total_df = pd.concat([total_df, df], axis=0)

	# 지연
	time.sleep(0.5)

In [3]:

# index 리셋
total_df = total_df.reset_index(drop=True)
# print(total_df)

In [ ]:
# 추출한 데이터 전처리 작업
# 중복 데이터 제거 : 제목이 중복되면 제거(하나만 남김)
clean_df = total_df.drop_duplicates(subset=['title'])

# 결측치 처리
clean_df = clean_df.dropna()

# 텍스트 길이 분석 후 처리 : AI 학습에서 짧은 문장은 정보량이 부족 => 제거
# 텍스트 길이 평균보다 큰 내용들만 추출

# 텍스트 길이의 평균 
mean_len = clean_df['title'].str.len().mean()

# 평균보다 큰 행들만 추출
final_df = clean_df.loc[clean_df['title'].str.len() > mean_len]

# 인덱스 리셋
final_df = final_df.reset_index(drop=True)
# 최종 : title(질문 일부), link(질문 링크)
# print(final_df)


                         title  \
0            실럽급여 받는 중 취업 햇을 시   
1           학사경고 취업,편입에 문제되나요?   
2      토스와 오픽 중 어떤 게 더 취업에...    
3    절도죄 기소유예 되었는데 어린이집 취업...    
4       국민취업지원제도는 어떻게 진행을 하나요?   
..                         ...   
96           취업시 범죄경력조회 동의v체크시   
97      실내디자인학원 비전공자도 취업까지...    
98     실업급여 구직활동 취업증명서 프린터...    
99    30대 남자 세무사 사무실 취업 가능합니까?   
100         국가국가장학금 취업연계중점대학유형   

                                                  link  
0    https://kin.naver.com/qna/detail.naver?d1id=6&...  
1    https://kin.naver.com/qna/detail.naver?d1id=11...  
2    https://kin.naver.com/qna/detail.naver?d1id=11...  
3    https://kin.naver.com/qna/detail.naver?d1id=6&...  
4    https://kin.naver.com/qna/detail.naver?d1id=11...  
..                                                 ...  
96   https://kin.naver.com/qna/detail.naver?d1id=6&...  
97   https://kin.naver.com/qna/detail.naver?d1id=11...  
98   https://kin.naver.com/qna/detail.naver?d1id=6&...  
99   https://kin.nave